# 🧬 Clase 2 — Registros Duplicados e Inconsistencias en Datos Financieros (Python)

## Aplicaciones de Minería de Datos a Economía y Finanzas
### Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

**Docente:** Dr. Darío Ezequiel Díaz · `drdarioezequieldiaz@gmail.com`

**Fecha de referencia:** Jueves 16 de abril de 2026

---

### 🎯 Objetivos del notebook

1. **Simular** un entorno financiero realista inyectando de forma controlada duplicados exactos, duplicados aproximados (*near-duplicates*) e inconsistencias sobre una base crediticia limpia.
2. **Detectar** duplicados exactos mediante `pandas.duplicated()` y cuantificar su impacto sobre estimadores estadísticos.
3. **Implementar** las métricas de similitud formalizadas en el apunte: distancia de Levenshtein, similitud de Jaro-Winkler y similitud coseno sobre n-gramas.
4. **Ejecutar** un pipeline de *probabilistic record linkage* inspirado en el modelo de Fellegi-Sunter (1969) con `recordlinkage`, incorporando *blocking* para escalabilidad.
5. **Validar** restricciones de dominio, dependencias funcionales y restricciones de integridad cruzada mediante *assertions* programáticas.
6. **Aplicar** detección estadística de inconsistencias: IQR contextualizado y z-score modificado basado en MAD (Iglewicz & Hoaglin, 1993).
7. **Consolidar** un protocolo secuencial de cinco etapas con un **log de auditoría** trazable, conforme al requerimiento regulatorio del BCRA y Basilea.

### 📚 Estrategia pedagógica

Partimos del **German Credit Dataset** (Statlog), una base crediticia íntegra con 1000 observaciones y **0% de valores faltantes**. Sobre ella construimos una capa sintética de atributos de identidad propios del contexto argentino (CUIT, nombre, provincia, fecha de otorgamiento, domicilio) e inyectamos luego duplicaciones e inconsistencias cuya estructura conocemos de antemano. Este diseño permite lo que resulta imposible con datos reales: **comparar cada detección contra la verdad de campo** y medir precisión, *recall* y sesgo con garantía.

> **Referencia teórica:** Apunte de Clase 2, Parte II — *Registros Duplicados e Inconsistencias en Datos Financieros* (Díaz, 2026).

---

## 1. Configuración del entorno

Cargamos las librerías necesarias. El *stack* se organiza en tres bloques:

| Bloque | Librerías | Función |
|---|---|---|
| **Base** | `pandas`, `numpy`, `matplotlib`, `seaborn` | Manipulación, cómputo y visualización |
| **Similitud de strings** | `rapidfuzz`, `jellyfish` | Levenshtein, Jaro-Winkler, ratios de similitud |
| **Record linkage** | `recordlinkage` | Blocking, comparación y clasificación Fellegi-Sunter |
| **Científicas** | `scipy`, `sklearn` | MAD, IQR, carga del dataset |

Las dos primeras librerías especializadas (`rapidfuzz` y `recordlinkage`) no forman parte del entorno por defecto de Google Colab, de modo que incluimos una instalación condicional. 

In [ ]:
# ══════════════════════════════════════════════════════════════
# 1. CONFIGURACIÓN DEL ENTORNO
# ══════════════════════════════════════════════════════════════

# --- Instalación condicional de librerías especializadas --------------
# Se instalan únicamente si no están presentes, evitando sobrescritura
# y acelerando la reejecución del notebook.
import importlib, subprocess, sys

def ensure(pkg, import_name=None):
    """Instala `pkg` si `import_name` no puede importarse."""
    name = import_name or pkg
    try:
        importlib.import_module(name)
    except ImportError:
        print(f"📦 Instalando {pkg} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("rapidfuzz")
ensure("jellyfish")
ensure("recordlinkage")

# --- Importaciones ----------------------------------------------------
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.datasets import fetch_openml
from scipy import stats

# Métricas de similitud
import jellyfish                                  # Jaro-Winkler nativo
from rapidfuzz import fuzz, distance              # Levenshtein, ratios

# Record linkage probabilístico
import recordlinkage
from recordlinkage.index import Block, SortedNeighbourhood
from recordlinkage.preprocessing import clean     # normalización de strings

# --- Configuración estética -------------------------------------------
warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.figsize"] = (13, 5)
plt.rcParams["figure.dpi"] = 100

# Paleta cromática del curso (Midnight Executive)
COL_NAVY   = "#1A2744"
COL_TEAL   = "#0C7C8A"
COL_AMBER  = "#F59E0B"
COL_RED    = "#E76F51"
COL_PURPLE = "#7C3AED"

# Reproducibilidad: fijamos la semilla global
RNG_SEED = 2026
np.random.seed(RNG_SEED)

print("✅ Entorno configurado correctamente")
print(f"   pandas {pd.__version__} · numpy {np.__version__}")
print(f"   recordlinkage {recordlinkage.__version__}")

## 2. Carga del dataset y enriquecimiento con atributos de identidad

El German Credit Dataset contiene atributos crediticios (monto, duración, historial, propósito) pero **carece de atributos de identidad** (nombre, CUIT, domicilio) que son justamente los más propensos a duplicación en la práctica bancaria argentina. Construimos entonces una capa sintética de identidad, cuidadosamente reproducible, que nos permitirá ejercitar todas las técnicas del apunte sobre un escenario verosímil.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2. CARGA DEL DATASET BASE
# ══════════════════════════════════════════════════════════════

# El dataset 'credit-g' en OpenML es el German Credit Dataset (Statlog).
# Cada fila representa una solicitud crediticia con 20 atributos + target.
german = fetch_openml(name="credit-g", version=1, as_frame=True, parser="auto")
df_base = german.frame.copy()

# --- Identificador crediticio único (clave candidata del negocio) ----
# Simulamos un ID de crédito propio de un core bancario: prefijo banco,
# año de otorgamiento y número correlativo de 6 dígitos.
df_base.insert(0, "id_credito",
               [f"BCR-2026-{i:06d}" for i in range(1, len(df_base) + 1)])

print(f"✅ Dataset cargado: {df_base.shape[0]} filas × {df_base.shape[1]} columnas")
print(f"   Valores faltantes totales: {df_base.isna().sum().sum()}")
print(f"\nVariable respuesta: distribución de 'class'")
print(df_base["class"].value_counts(normalize=True).round(3).to_string())

### 2.1 Construcción de la capa sintética de identidad

Generamos nombres, CUIT, provincias, fechas y domicilios que respetan la lógica del negocio financiero argentino. La clave conceptual es que **cada `id_credito` apunta a una única persona jurídica/física**, de modo que cualquier duplicación posterior será artificial y conocida.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 2.1 ENRIQUECIMIENTO SINTÉTICO: IDENTIDAD Y GEOGRAFÍA
# ══════════════════════════════════════════════════════════════

rng = np.random.default_rng(RNG_SEED)

# --- Diccionarios de referencia (verosimilitud argentina) ------------
NOMBRES = ["Juan Carlos", "María Fernanda", "Roberto", "Lucía", "Diego",
           "Sofía", "Martín", "Valentina", "Alejandro", "Camila",
           "Federico", "Agustina", "Sebastián", "Florencia", "Gonzalo",
           "Julieta", "Matías", "Carolina", "Nicolás", "Paula"]

APELLIDOS = ["González", "Rodríguez", "Fernández", "López", "Martínez",
             "Sánchez", "Pérez", "García", "Romero", "Sosa",
             "Torres", "Álvarez", "Ruiz", "Ramírez", "Flores",
             "Acosta", "Benítez", "Medina", "Suárez", "Herrera"]

PROVINCIAS_CP = {
    # provincia : (rango_cp_inicial, rango_cp_final)
    "CABA":       (1000, 1499),
    "Buenos Aires": (1500, 8000),
    "Santa Fe":   (2000, 3099),
    "Córdoba":    (5000, 5999),
    "Mendoza":    (5500, 5599),
    "Misiones":   (3300, 3399),
    "Tucumán":    (4000, 4199),
    "Salta":      (4400, 4499),
}
PROV_LIST = list(PROVINCIAS_CP.keys())

TIPOS_EMPLEO = ["empleado_publico", "empleado_privado", "autonomo",
                "profesional", "jubilado"]

def generar_cuit(rng):
    """Genera CUIT con formato 20-DNI-V (tipo 20 = persona física masculina,
    27 = femenina, 30/33 = jurídica). Sin cálculo real de dígito verificador."""
    prefijo = rng.choice([20, 23, 24, 27])
    dni = rng.integers(10_000_000, 45_000_000)
    verif = rng.integers(0, 10)
    return f"{prefijo}-{dni:08d}-{verif}"

def generar_cp(prov, rng):
    lo, hi = PROVINCIAS_CP[prov]
    return str(rng.integers(lo, hi + 1)).zfill(4)

# --- Construcción vectorizada de la capa de identidad ----------------
n = len(df_base)

df_base["nombre"]    = [f"{rng.choice(NOMBRES)} {rng.choice(APELLIDOS)}" for _ in range(n)]
df_base["cuit"]      = [generar_cuit(rng) for _ in range(n)]
df_base["provincia"] = rng.choice(PROV_LIST, size=n, p=[0.33, 0.29, 0.08, 0.10,
                                                         0.05, 0.06, 0.05, 0.04])
df_base["cod_postal"] = [generar_cp(p, rng) for p in df_base["provincia"]]
df_base["tipo_empleo"] = rng.choice(TIPOS_EMPLEO, size=n)

# --- Variables financieras derivadas (coherentes con credit_amount) --
# Ingreso mensual: correlaciona positivamente con monto solicitado.
# Transformamos credit_amount (DEM de los '90) a escala de pesos actuales
# preservando la distribución relativa.
escala_ars = 250_000   # factor de conversión didáctico
df_base["ingreso_mensual"] = (
    df_base["credit_amount"] * escala_ars / df_base["duration"]
    * rng.uniform(0.8, 1.2, size=n)
).round(-2).astype(int)

# Fechas de otorgamiento: uniformes en [2023-01-01, 2026-04-15]
inicio = pd.Timestamp("2023-01-01")
fin    = pd.Timestamp("2026-04-15")
dias_rango = (fin - inicio).days
df_base["fecha_otorgamiento"] = [
    inicio + pd.Timedelta(days=int(d))
    for d in rng.integers(0, dias_rango, size=n)
]

# Fecha de vencimiento = otorgamiento + duration (meses)
df_base["fecha_vencimiento"] = df_base.apply(
    lambda r: r["fecha_otorgamiento"] + pd.DateOffset(months=int(r["duration"])),
    axis=1
)

# Conservamos un subconjunto de columnas para el resto del notebook.
cols_finales = ["id_credito", "nombre", "cuit", "provincia", "cod_postal",
                "tipo_empleo", "ingreso_mensual", "credit_amount", "duration",
                "fecha_otorgamiento", "fecha_vencimiento", "class"]
df_limpio = df_base[cols_finales].copy()

print(f"✅ Capa de identidad generada. Dataset limpio: {df_limpio.shape}")
print("\nPrimeras 5 filas del dataset enriquecido:")
df_limpio.head()

### 📝 Interpretación

Obtenemos un dataset crediticio de **1000 registros íntegros** sobre el que podemos enunciar cinco propiedades axiomáticas que serán violadas deliberadamente en la siguiente sección:

- Unicidad: todo `id_credito` aparece exactamente una vez.
- Dependencia funcional: `cuit → nombre` (un CUIT pertenece a una única persona).
- Dependencia funcional: `cod_postal → provincia` (un CP fija unívocamente la provincia).
- Integridad temporal: `fecha_otorgamiento < fecha_vencimiento`.
- Plausibilidad económica: el ingreso mensual guarda una relación razonable con el monto solicitado (ratio cuota/ingreso).

Al partir de esta línea base controlada, toda desviación posterior podrá imputarse con certeza a la inyección controlada que realizamos a continuación, no a artefactos desconocidos.

## 3. Inyección controlada de duplicados e inconsistencias

Construimos `df_sucio` introduciendo tres patologías cuya estructura conocemos:

1. **Duplicados exactos** (50 registros): copias textuales bit a bit.
2. **Duplicados aproximados** (40 registros): variantes ortográficas, abreviaturas, cambios de caja.
3. **Inconsistencias** (≈ 80 registros distribuidas en varios tipos): violaciones de dominio, de dependencias funcionales y de integridad cruzada.

El dataset final tendrá más filas que el original, lo que ya anticipa el sesgo en la estimación de frecuencias formalizado por el **Teorema 5.1** del apunte.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 3. INYECCIÓN CONTROLADA DE PATOLOGÍAS
# ══════════════════════════════════════════════════════════════

df_sucio = df_limpio.copy()
rng = np.random.default_rng(RNG_SEED + 1)

# ---------------------------------------------------------------
# 3.1 Duplicados exactos (50 copias bit a bit)
# ---------------------------------------------------------------
idx_exactos = rng.choice(df_sucio.index, size=50, replace=False)
dup_exactos = df_sucio.loc[idx_exactos].copy()
dup_exactos["_patologia"] = "dup_exacto"
df_sucio["_patologia"] = "limpio"


# ---------------------------------------------------------------
# 3.2 Duplicados aproximados (40 variantes ortográficas)
# ---------------------------------------------------------------
idx_aprox = rng.choice(
    [i for i in df_sucio.index if i not in idx_exactos],
    size=40, replace=False
)
dup_aprox = df_sucio.loc[idx_aprox].copy()
dup_aprox["_patologia"] = "dup_aprox"

def variar_nombre(nombre, rng):
    """Aplica transformaciones que preservan la identidad semántica:
    quita acentos, cambia caja, abrevia el nombre de pila, introduce
    un error tipográfico simple. Sintetiza los patrones observados
    en el Ejemplo 3.1 del apunte (variantes de 'Gonzalez, María F.')."""
    transformacion = rng.choice(["sin_tildes", "mayusculas",
                                 "abreviado", "typo", "con_coma"])
    reemplazos = str.maketrans("áéíóúÁÉÍÓÚ", "aeiouAEIOU")
    if transformacion == "sin_tildes":
        return nombre.translate(reemplazos)
    if transformacion == "mayusculas":
        return nombre.translate(reemplazos).upper()
    if transformacion == "abreviado":
        partes = nombre.split()
        if len(partes) >= 2:
            return f"{partes[0][0]}. {' '.join(partes[1:])}"
        return nombre
    if transformacion == "typo":
        pos = rng.integers(1, max(2, len(nombre) - 1))
        return nombre[:pos] + nombre[pos + 1:]   # eliminación de 1 carácter
    if transformacion == "con_coma":
        partes = nombre.split()
        if len(partes) >= 2:
            return f"{partes[-1]}, {' '.join(partes[:-1])}".upper()
        return nombre
    return nombre

# Variamos también el formato del CUIT (con/sin guiones)
def variar_cuit(cuit, rng):
    if rng.random() < 0.5:
        return cuit.replace("-", "")        # sin guiones
    return cuit

dup_aprox["nombre"] = [variar_nombre(n, rng) for n in dup_aprox["nombre"]]
dup_aprox["cuit"]   = [variar_cuit(c, rng)  for c in dup_aprox["cuit"]]

# A los aproximados les cambiamos también el id_credito (entraron por otra vía)
dup_aprox["id_credito"] = [f"BCR-ALT-{i:06d}"
                           for i in range(1, len(dup_aprox) + 1)]

# Concatenamos
df_sucio = pd.concat([df_sucio, dup_exactos, dup_aprox], ignore_index=True)

print(f"Registros tras inyectar duplicados: {len(df_sucio):,}")
print(f"  · limpios     : {(df_sucio['_patologia']=='limpio').sum()}")
print(f"  · dup_exactos : {(df_sucio['_patologia']=='dup_exacto').sum()}")
print(f"  · dup_aprox   : {(df_sucio['_patologia']=='dup_aprox').sum()}")

In [ ]:
# ---------------------------------------------------------------
# 3.3 Inconsistencias (tres tipologías)
# ---------------------------------------------------------------
# Trabajamos sobre copia para no ensuciar los duplicados ya insertados.
idx_validos = df_sucio.index.tolist()

# A) Violación de dominio: ingreso negativo y edad-equivalente absurda
idx_dom = rng.choice(idx_validos, size=20, replace=False)
df_sucio.loc[idx_dom[:10], "ingreso_mensual"] = -rng.integers(
    50_000, 500_000, size=10
)
# Ingresos gigantescos (probable error de unidades: dólares en lugar de pesos)
df_sucio.loc[idx_dom[10:], "ingreso_mensual"] = rng.integers(
    10_000_000_000, 50_000_000_000, size=10
)
df_sucio.loc[idx_dom, "_patologia"] = "incons_dominio"

# B) Violación de dependencia funcional: cod_postal incoherente con provincia
idx_dep = rng.choice(
    [i for i in idx_validos if i not in idx_dom], size=25, replace=False
)
# Asignamos un CP que corresponde a OTRA provincia.
for i in idx_dep:
    prov_actual = df_sucio.at[i, "provincia"]
    otras = [p for p in PROV_LIST if p != prov_actual]
    otra_prov = rng.choice(otras)
    df_sucio.at[i, "cod_postal"] = generar_cp(otra_prov, rng)
df_sucio.loc[idx_dep, "_patologia"] = "incons_dep_func"

# C) Violación de integridad cruzada: fecha_venc < fecha_otorg
idx_int = rng.choice(
    [i for i in idx_validos if i not in idx_dom and i not in idx_dep],
    size=20, replace=False
)
for i in idx_int:
    df_sucio.at[i, "fecha_vencimiento"] = (
        df_sucio.at[i, "fecha_otorgamiento"] - pd.DateOffset(months=6)
    )
df_sucio.loc[idx_int, "_patologia"] = "incons_integridad"

# D) Violación de dependencia funcional CUIT → nombre
# Dos registros con el mismo CUIT pero nombres distintos
idx_cuit = rng.choice(
    [i for i in idx_validos if i not in idx_dom
     and i not in idx_dep and i not in idx_int],
    size=15, replace=False
)
# Tomamos pares y les asignamos el mismo CUIT pero nombres diferentes
for k in range(0, len(idx_cuit) - 1, 2):
    a, b = idx_cuit[k], idx_cuit[k + 1]
    df_sucio.at[b, "cuit"] = df_sucio.at[a, "cuit"]
df_sucio.loc[idx_cuit, "_patologia"] = "incons_cuit_nombre"

print(f"✅ Inconsistencias inyectadas: "
      f"{(df_sucio['_patologia'].str.startswith('incons')).sum()}")
print("\nResumen del dataset sucio:")
print(df_sucio["_patologia"].value_counts().to_string())
print(f"\nTotal de registros 'sucios': {len(df_sucio):,}")

### 📝 Interpretación

El dataset `df_sucio` reproduce con fidelidad las patologías documentadas en el apunte. La columna `_patologia` actúa como **etiqueta de ground truth** que, pedagógicamente, nos permitirá evaluar cada método de detección con métricas de clasificación. En producción tal etiqueta no existe; todo el esfuerzo de detección busca precisamente reconstruirla.

> ⚠️ En escenarios reales no dispondremos de esta columna. La estrategia didáctica consiste en inyectar patologías conocidas sobre un dataset íntegro para medir la eficacia de cada algoritmo contra una verdad que sólo nosotros conocemos.

---

# 🧩 PARTE I — REGISTROS DUPLICADOS

> *"En toda base de datos financiera subyace un principio fundamental: cada entidad del mundo real debe corresponder a exactamente un registro."* (Apunte, §1)

---

## 4. Detección de duplicados exactos

Los duplicados exactos son el caso trivial: coinciden en **todos los atributos**. Su detección es $O(n \log n)$ mediante *hashing*, implementado nativamente en `pandas.duplicated()`. Pese a su simplicidad, constituyen el punto de partida obligatorio de cualquier pipeline de calidad: si los dejamos pasar, las métricas posteriores quedan contaminadas.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4. DETECCIÓN DE DUPLICADOS EXACTOS
# ══════════════════════════════════════════════════════════════

# Para la detección de duplicados exactos ignoramos id_credito y la
# etiqueta de ground truth (_patologia), pues son metadatos que por
# definición difieren entre registros duplicados insertados.
cols_negocio = [c for c in df_sucio.columns
                if c not in ("id_credito", "_patologia")]

mask_exactos = df_sucio.duplicated(subset=cols_negocio, keep=False)
n_exactos = mask_exactos.sum()

print(f"🔎 Filas involucradas en duplicación exacta: {n_exactos}")
print(f"   (incluye originales + copias)")

# Comparación contra ground truth
true_exactos = (df_sucio["_patologia"] == "dup_exacto").sum()
print(f"\nGround truth: {true_exactos} copias inyectadas")
print(f"Detección dice: {n_exactos - true_exactos} originales + "
      f"{true_exactos} copias = {n_exactos} filas marcadas")

# Evaluación: los 50 duplicados exactos deben aparecer dentro de las
# filas marcadas. El recall sobre la clase 'dup_exacto' debe ser 1.0.
recall_exacto = mask_exactos[df_sucio["_patologia"]=="dup_exacto"].mean()
print(f"\n✅ Recall sobre duplicados exactos: {recall_exacto:.2%}")

### 4.1 Impacto cuantitativo sobre estimadores

El **Teorema 5.1** del apunte establece que la frecuencia empírica bajo duplicación está sesgada cuando la multiplicidad $c_i$ correlaciona con el atributo. Veamos en la práctica qué ocurre con estadísticos clave de la cartera antes y después de la deduplicación exacta.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 4.1 IMPACTO DE LA DUPLICACIÓN SOBRE ESTIMADORES
# ══════════════════════════════════════════════════════════════

df_dedup_exacto = df_sucio.drop_duplicates(subset=cols_negocio, keep="first").copy()

metricas = pd.DataFrame({
    "Con duplicados": [
        len(df_sucio),
        df_sucio["credit_amount"].mean(),
        df_sucio["ingreso_mensual"].median(),
        (df_sucio["class"] == "bad").mean(),
        df_sucio["provincia"].value_counts(normalize=True).max(),
    ],
    "Sin duplicados exactos": [
        len(df_dedup_exacto),
        df_dedup_exacto["credit_amount"].mean(),
        df_dedup_exacto["ingreso_mensual"].median(),
        (df_dedup_exacto["class"] == "bad").mean(),
        df_dedup_exacto["provincia"].value_counts(normalize=True).max(),
    ]
}, index=["N registros", "Media credit_amount",
          "Mediana ingreso", "Tasa default (bad)",
          "Máx. concentración provincial"])

metricas["Δ relativo (%)"] = (
    (metricas["Sin duplicados exactos"] - metricas["Con duplicados"])
    / metricas["Con duplicados"] * 100
).round(3)

print("📊 Impacto de la deduplicación exacta sobre estadísticos clave\n")
print(metricas.round(3).to_string())

In [ ]:
# --- Visualización comparativa de distribuciones ---------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Subplot 1: histograma de credit_amount
axes[0].hist(df_sucio["credit_amount"], bins=40, alpha=0.55,
             label="Con duplicados", color=COL_RED, edgecolor="white")
axes[0].hist(df_dedup_exacto["credit_amount"], bins=40, alpha=0.75,
             label="Sin duplicados exactos", color=COL_TEAL, edgecolor="white")
axes[0].axvline(df_sucio["credit_amount"].mean(), color=COL_RED,
                linestyle="--", lw=1.5, label="media sucia")
axes[0].axvline(df_dedup_exacto["credit_amount"].mean(), color=COL_TEAL,
                linestyle="--", lw=1.5, label="media limpia")
axes[0].set_title("Distribución de credit_amount", color=COL_NAVY, fontweight="bold")
axes[0].set_xlabel("Monto del crédito"); axes[0].legend()

# Subplot 2: barras de tasa default por provincia
tasas_sucio = df_sucio.groupby("provincia")["class"].apply(
    lambda s: (s == "bad").mean()).sort_values()
tasas_limpio = df_dedup_exacto.groupby("provincia")["class"].apply(
    lambda s: (s == "bad").mean()).reindex(tasas_sucio.index)

x = np.arange(len(tasas_sucio))
w = 0.38
axes[1].bar(x - w/2, tasas_sucio.values, w,
            label="Con duplicados", color=COL_RED, alpha=0.85)
axes[1].bar(x + w/2, tasas_limpio.values, w,
            label="Sin duplicados", color=COL_TEAL, alpha=0.85)
axes[1].set_xticks(x); axes[1].set_xticklabels(tasas_sucio.index, rotation=45)
axes[1].set_title("Tasa de default por provincia",
                  color=COL_NAVY, fontweight="bold")
axes[1].set_ylabel("Prop. 'bad'"); axes[1].legend()

plt.tight_layout(); plt.show()

### 📝 Interpretación

El impacto numérico depende de **qué correlaciona con la multiplicidad**. Si los duplicados se distribuyen de manera puramente aleatoria sobre la cartera, las medias y proporciones se preservan (apenas cambia $N$). Si en cambio los duplicados se concentran en subpoblaciones específicas —por ejemplo, clientes que solicitan crédito simultáneamente en varias sucursales, típicamente de perfil más riesgoso— entonces la **tasa de default se infla artificialmente**, el modelo de scoring entrenado sobre tal muestra produce probabilidades de incumplimiento sobrestimadas y la política crediticia resulta excesivamente conservadora. Este fenómeno es exactamente el advertido en el recuadro "Sesgo en scoring crediticio" del apunte.

## 5. Métricas de similitud para duplicados aproximados

Los duplicados exactos se extinguen con `drop_duplicates()`; los aproximados exigen **comparación difusa** (*fuzzy matching*). Implementamos las tres métricas formalizadas en el apunte:

- **Levenshtein**: distancia de edición; mide cuántas inserciones/eliminaciones/sustituciones separan dos strings.
- **Jaro-Winkler**: penaliza los desajustes pero premia los prefijos comunes (especialmente apropiado para nombres propios).
- **Similitud coseno sobre bigramas**: robusta frente a reordenamientos de palabras.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 5. MÉTRICAS DE SIMILITUD: CATÁLOGO COMPARATIVO
# ══════════════════════════════════════════════════════════════

# Cuatro strings del Ejemplo 3.1 del apunte (variantes del mismo nombre)
referencia = "González, María Fernanda"
variantes = [
    "Gonzalez Maria Fernanda",      # formulario web
    "GONZALES, MARIA F.",           # sucursal (typo + abreviatura)
    "Gonsales, Ma. Fernanda",       # call center
    "GONZALEZ MARIA FERNANDA",      # bureau de crédito
    "Rodríguez, María Fernanda",    # ¡persona distinta! Control negativo.
    "María Fernanda González",      # reordenamiento
]

def bigram_cosine(a, b):
    """Similitud coseno sobre bigramas (n=2) de caracteres."""
    def bigrams(s):
        s = s.lower()
        return [s[i:i+2] for i in range(len(s) - 1)]
    from collections import Counter
    ca, cb = Counter(bigrams(a)), Counter(bigrams(b))
    comunes = set(ca) & set(cb)
    dot = sum(ca[g] * cb[g] for g in comunes)
    na = np.sqrt(sum(v**2 for v in ca.values()))
    nb = np.sqrt(sum(v**2 for v in cb.values()))
    return dot / (na * nb) if na * nb > 0 else 0.0

tabla = []
for v in variantes:
    tabla.append({
        "Variante": v,
        "Levenshtein (distancia)": distance.Levenshtein.distance(referencia, v),
        "Levenshtein (sim. norm.)": 1 - distance.Levenshtein.normalized_distance(
            referencia, v),
        "Jaro-Winkler": jellyfish.jaro_winkler_similarity(referencia, v),
        "Coseno bigramas": bigram_cosine(referencia, v),
    })
comp = pd.DataFrame(tabla)
print(f"📋 Cadena de referencia: '{referencia}'\n")
print(comp.round(3).to_string(index=False))

In [ ]:
# --- Visualización: mapa de calor de métricas -----------------------
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    comp.set_index("Variante")[["Levenshtein (sim. norm.)",
                                "Jaro-Winkler", "Coseno bigramas"]],
    annot=True, fmt=".3f", cmap="YlGnBu", vmin=0, vmax=1,
    cbar_kws={"label": "Similitud"}, ax=ax
)
ax.set_title("Similitud entre variantes y cadena de referencia",
             color=COL_NAVY, fontweight="bold")
plt.tight_layout(); plt.show()

### 📝 Interpretación

Tres observaciones medulares:

1. **Jaro-Winkler domina para variantes conservadoras del nombre**. Las cinco primeras filas (que son duplicados aproximados legítimos) superan consistentemente el umbral $0{,}85$ recomendado en el apunte, mientras que el control negativo —"Rodríguez"— cae por debajo. Esto confirma la bondad de Jaro-Winkler para el dominio de nombres propios: capta bien los errores al final del string que son los más frecuentes en la digitación.

2. **El coseno sobre bigramas tolera reordenamientos**. La sexta fila ("María Fernanda González") exhibe Jaro-Winkler bajo (por el cambio de orden entre apellido y nombre) pero similitud coseno elevada, porque los bigramas comunes no dependen de la posición. En bases donde el formato de nombre varía entre canales, esta métrica resulta indispensable.

3. **Ninguna métrica es universalmente superior**. La práctica estándar consiste en combinar varias (concatenadas como atributos del vector de comparación $\boldsymbol{\gamma}$ del modelo Fellegi-Sunter) y aprender pesos óptimos. Es el enfoque que formalizamos a continuación.

## 6. Blocking y record linkage probabilístico (Fellegi-Sunter)

La comparación exhaustiva de todos los pares posibles en una base de $n$ registros requiere $\binom{n}{2}$ operaciones: con nuestros $\sim 1090$ registros serían ${\sim}594.000$ comparaciones, manejable; pero en una cartera de 500.000 clientes el costo asciende a $\sim 1{,}25 \times 10^{11}$ pares, computacionalmente prohibitivo.

La estrategia de ***blocking*** particiona el dataset y sólo compara registros dentro del mismo bloque. Utilizamos como clave de bloqueo las primeras tres letras del nombre (normalizado), un equilibrio razonable entre *recall* y eficiencia.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 6. BLOCKING + PROBABILISTIC RECORD LINKAGE
# ══════════════════════════════════════════════════════════════

# --- Preparación: normalización de atributos de matching -------------
df_rl = df_sucio.reset_index(drop=True).copy()
df_rl["nombre_norm"] = clean(df_rl["nombre"])      # mayúsculas, sin acentos
df_rl["cuit_norm"]   = df_rl["cuit"].str.replace("-", "", regex=False)
df_rl["cp_norm"]     = df_rl["cod_postal"].astype(str)

# Clave de bloqueo: 3 primeros caracteres del nombre normalizado
df_rl["block_key"] = df_rl["nombre_norm"].str[:3]

# --- Indexación: generación del conjunto candidato de pares ----------
indexer = recordlinkage.Index()
indexer.block("block_key")                 # blocking exacto por prefijo de 3
candidatos = indexer.index(df_rl)

n_total_pares = len(df_rl) * (len(df_rl) - 1) // 2
print(f"📦 Total teórico de pares:   {n_total_pares:,}")
print(f"📦 Pares tras blocking:      {len(candidatos):,}")
print(f"📦 Reducción:                {100*(1-len(candidatos)/n_total_pares):.2f}%")

In [ ]:
# --- Comparación: construcción del vector γ --------------------------
comparer = recordlinkage.Compare()

# Atributo por atributo con su métrica idónea
comparer.string("nombre_norm", "nombre_norm",
                method="jarowinkler", threshold=0.85, label="nombre_jw")
comparer.string("cuit_norm",   "cuit_norm",
                method="levenshtein", threshold=0.90, label="cuit_lev")
comparer.exact("cp_norm", "cp_norm",             label="cp_eq")
comparer.exact("provincia", "provincia",         label="prov_eq")
comparer.numeric("credit_amount", "credit_amount",
                 method="gauss", offset=100, scale=500, label="monto_gauss")

features = comparer.compute(candidatos, df_rl)
print(f"✅ Vector γ calculado para {len(features):,} pares candidatos")
print("\nDistribución del score total por número de atributos coincidentes:")
print(features.sum(axis=1).value_counts().sort_index().to_string())

### 6.1 Clasificación Fellegi-Sunter simplificada

El modelo Fellegi-Sunter pleno requiere estimar las $m$-probabilities y $u$-probabilities con EM sobre los datos, procedimiento que excede el alcance de esta clase. Empleamos en su lugar un **clasificador heurístico** basado en umbrales sobre el score agregado: suficiente para ilustrar la mecánica y obtener buen recall en nuestro escenario controlado.

In [ ]:
# --- Clasificación por umbral ---------------------------------------
score = features.sum(axis=1)

# Umbrales (parametrizables en producción vía calibración)
T_mu, T_lambda = 3.5, 2.0   # match / posible match / no match

clasificacion = pd.Series("no_match", index=score.index)
clasificacion[score >= T_mu]                       = "match"
clasificacion[(score > T_lambda) & (score < T_mu)] = "posible_match"

print("📊 Distribución de decisiones:")
print(clasificacion.value_counts().to_string())

matches = clasificacion[clasificacion == "match"].index

In [ ]:
# --- Evaluación contra el ground truth -------------------------------
# Reconstruimos el ground truth: qué pares SON el mismo cliente?
# En nuestra construcción, un duplicado (exacto o aproximado) comparte
# todas las variables crediticias con SU ORIGINAL, salvo id_credito
# y el nombre en el caso aproximado. Identificamos pares reales con:

def es_par_verdadero(i, j, df):
    a, b = df.loc[i], df.loc[j]
    if a["_patologia"] == "dup_exacto" or b["_patologia"] == "dup_exacto":
        # exactos: coinciden en cuit + monto + duracion + fecha
        return (a["cuit"] == b["cuit"] and
                a["credit_amount"] == b["credit_amount"] and
                a["duration"] == b["duration"])
    if a["_patologia"] == "dup_aprox" or b["_patologia"] == "dup_aprox":
        return (a["credit_amount"] == b["credit_amount"] and
                a["duration"] == b["duration"] and
                a["fecha_otorgamiento"] == b["fecha_otorgamiento"])
    return False

# Eficientemente: marcamos como positivos los pares del índice candidato
# cuyos dos miembros son "duplicado" de algún tipo y comparten monto+duración.
df_rl_pat = df_rl[["_patologia", "cuit", "credit_amount",
                   "duration", "fecha_otorgamiento"]]

y_true = [es_par_verdadero(i, j, df_rl_pat) for i, j in candidatos]
y_pred = clasificacion == "match"

from sklearn.metrics import classification_report, confusion_matrix
print("📊 Matriz de confusión (sobre pares candidatos):")
cm = confusion_matrix(y_true, y_pred)
print(pd.DataFrame(cm, index=["verdadero_no", "verdadero_sí"],
                   columns=["pred_no", "pred_sí"]))
print("\n", classification_report(y_true, y_pred,
                                  target_names=["no_match", "match"],
                                  digits=3))

### 📝 Interpretación

El pipeline completo combina **blocking + comparación multicampo + umbralización**. Tres lecciones:

- **El blocking reduce órdenes de magnitud** el costo computacional con pérdida acotada de recall: los duplicados cuyo nombre difiere en los primeros tres caracteres escapan (por ejemplo, un typo precisamente en el prefijo). Por eso en producción se emplean **múltiples claves de bloqueo** (apellido inicial, año de nacimiento, últimos dígitos del DNI) en disyunción: un par se evalúa si coincide en al menos una.
- **El umbral $T_\mu$ regula la precisión** a costa de recall. Un umbral laxo captura más duplicados pero genera falsos positivos que en finanzas son costosos: fusionar erróneamente dos clientes produce la propagación de información crediticia entre personas distintas, con implicancias regulatorias.
- **La zona "posible match"** debe revisarse manualmente. En bancos grandes esta revisión se tercerizar equipos de *data stewardship* apoyados en herramientas de aprendizaje activo como la librería `dedupe`.

## 7. Consolidación del *golden record*

Una vez identificados los clusters de registros que pertenecen a la misma entidad, corresponde sintetizar un **registro dorado** (*golden record*) por entidad. La regla de consolidación depende del atributo:

- **Atributos textuales**: tomar la versión más frecuente o la más reciente.
- **Atributos numéricos**: promediar, maximizar, o tomar el valor de la fuente más confiable.
- **Fechas**: priorizar la más reciente (principio de *recency*).

In [ ]:
# ══════════════════════════════════════════════════════════════
# 7. CONSTRUCCIÓN DEL GOLDEN RECORD
# ══════════════════════════════════════════════════════════════

# Convertimos los pares 'match' en clusters mediante Union-Find.
class UnionFind:
    def __init__(self, n):
        self.parent = list(range(n))
    def find(self, x):
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]
            x = self.parent[x]
        return x
    def union(self, x, y):
        rx, ry = self.find(x), self.find(y)
        if rx != ry:
            self.parent[rx] = ry

uf = UnionFind(len(df_rl))
for i, j in matches:
    uf.union(i, j)

df_rl["cluster_id"] = [uf.find(i) for i in range(len(df_rl))]

# Regla de consolidación: primero por cluster, tomando el registro más
# reciente (mayor fecha_otorgamiento) como base.
def consolidar(grupo):
    base = grupo.sort_values("fecha_otorgamiento", ascending=False).iloc[0].copy()
    base["n_fuentes"] = len(grupo)          # trazabilidad: cuántos originaron
    base["ids_originales"] = "|".join(grupo["id_credito"].astype(str))
    return base

golden = df_rl.groupby("cluster_id", group_keys=False).apply(consolidar)

print(f"Registros sucios de entrada:   {len(df_rl):,}")
print(f"Clusters identificados:        {df_rl['cluster_id'].nunique():,}")
print(f"Golden records producidos:     {len(golden):,}")
print(f"Reducción de cardinalidad:     "
      f"{100*(1 - len(golden)/len(df_rl)):.2f}%")

# Ejemplo de un cluster grande (si existe)
clusters_grandes = df_rl.groupby("cluster_id").size()
cluster_destacado = clusters_grandes.idxmax()
print(f"\n🔬 Cluster más grande (id={cluster_destacado}, "
      f"tamaño={clusters_grandes.max()}):")
df_rl[df_rl["cluster_id"] == cluster_destacado][
    ["id_credito", "nombre", "cuit", "_patologia"]
].head(10)

### 📝 Interpretación

El *golden record* es el resultado tangible de todo el pipeline de record linkage: de los ~1090 registros sucios obtenemos ~1000 entidades únicas (idealmente recuperamos el $N$ original). Cada registro dorado conserva metadatos de trazabilidad (`n_fuentes`, `ids_originales`) que permiten auditar la decisión de fusión, requisito ineludible en modelos regulados.

---

# ⚠️ PARTE II — INCONSISTENCIAS EN DATOS FINANCIEROS

> *"Una base de datos es consistente cuando todo registro satisface simultáneamente las restricciones de dominio, las reglas de integridad referencial, las dependencias funcionales y las reglas de negocio que gobiernan el fenómeno modelado."* (Apunte, §8)

Trabajamos sobre `df_dedup_exacto` (ya liberado de duplicación exacta) para aplicar las cuatro capas de validación del protocolo del apunte.

---

## 8. Etapa 1 — Validación de dominio

Para cada atributo $A_j$ definimos su dominio admisible $\mathcal{D}_j$ y marcamos las violaciones.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 8. VALIDACIÓN DE DOMINIO
# ══════════════════════════════════════════════════════════════

df_ins = df_dedup_exacto.copy()   # base para detección de inconsistencias

# --- Definición explícita de los dominios (diccionario de reglas) ----
REGLAS_DOMINIO = {
    "ingreso_mensual": {
        "descripcion": "Ingreso mensual en ARS",
        "valida": lambda x: (x >= 0) & (x < 1e9),   # no negativo, no absurdo
    },
    "credit_amount": {
        "descripcion": "Monto del crédito (positivo)",
        "valida": lambda x: x > 0,
    },
    "duration": {
        "descripcion": "Duración en meses (1 a 120)",
        "valida": lambda x: (x >= 1) & (x <= 120),
    },
    "cod_postal": {
        "descripcion": "Código postal (4 dígitos numéricos)",
        "valida": lambda x: x.astype(str).str.match(r"^\d{4}$"),
    },
    "provincia": {
        "descripcion": "Provincia en conjunto válido",
        "valida": lambda x: x.isin(PROV_LIST),
    },
    "tipo_empleo": {
        "descripcion": "Tipo de empleo válido",
        "valida": lambda x: x.isin(TIPOS_EMPLEO),
    },
}

# --- Evaluación de cada regla ---------------------------------------
log_dominio = []
for col, regla in REGLAS_DOMINIO.items():
    mask_valida = regla["valida"](df_ins[col])
    n_viol = (~mask_valida).sum()
    log_dominio.append({
        "variable": col,
        "descripcion": regla["descripcion"],
        "violaciones": n_viol,
        "pct": f"{100*n_viol/len(df_ins):.2f}%"
    })

df_log_dom = pd.DataFrame(log_dominio)
print("🔍 Log de validación de dominio:\n")
print(df_log_dom.to_string(index=False))

In [ ]:
# --- Inspección de casos patológicos concretos ----------------------
mask_ingreso_negativo = df_ins["ingreso_mensual"] < 0
mask_ingreso_absurdo  = df_ins["ingreso_mensual"] > 1e9

print(f"Ingresos negativos:  {mask_ingreso_negativo.sum()}")
print(f"Ingresos absurdos (>1B): {mask_ingreso_absurdo.sum()}")

print("\n📋 Muestra de ingresos negativos (probable error de signo):")
cols_muestra = ["id_credito", "nombre", "cuit", "ingreso_mensual",
                "credit_amount", "_patologia"]
df_ins.loc[mask_ingreso_negativo, cols_muestra].head()

### 📝 Interpretación

Las violaciones de dominio son el tipo más "limpio" de inconsistencia: su detección es determinista, su tratamiento suele ser inmediato. Dos cursos de acción típicos:

- **Corrección por regla heurística**: un ingreso negativo a menudo es un error de signo; puede revertirse automáticamente tomando $|x|$, con registro en el log de auditoría.
- **Marcado como `NA`**: un ingreso mayor a $10^9$ ARS mensuales es inverificable; lo prudente es anularlo y tratar como valor faltante en etapas posteriores del pipeline, conectando con las técnicas de imputación vistas en la clase de valores faltantes.

> ⚠️ **Nunca corregir silenciosamente.** Toda transformación debe quedar trazada con `(valor_original, regla_violada, acción, justificación)`, conforme al principio de trazabilidad del apunte.

## 9. Etapa 2 — Dependencias funcionales

Una dependencia funcional $A \to B$ exige que todo valor de $A$ determine un único valor de $B$. Verificamos dos dependencias críticas:

- $\text{cod\_postal} \to \text{provincia}$: un CP define unívocamente la provincia.
- $\text{cuit} \to \text{nombre}$: un CUIT corresponde a una única persona.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 9. DEPENDENCIAS FUNCIONALES
# ══════════════════════════════════════════════════════════════

def violaciones_df(df, determinante, dependiente):
    """
    Retorna:
      - violaciones: número de valores del determinante que se asocian
        con más de un valor del dependiente.
      - detalle: DataFrame con los valores problemáticos.
    """
    grupos = (df.groupby(determinante)[dependiente]
                .nunique()
                .sort_values(ascending=False))
    problematicos = grupos[grupos > 1]
    return len(problematicos), problematicos

# --- Dependencia 1: cod_postal → provincia --------------------------
n1, det1 = violaciones_df(df_ins, "cod_postal", "provincia")
print(f"🔸 cod_postal → provincia")
print(f"   Violaciones: {n1} CP con >1 provincia")
if n1 > 0:
    print(f"   Top 5 CP con mayor dispersión:\n{det1.head().to_string()}")

# --- Dependencia 2: cuit → nombre -----------------------------------
n2, det2 = violaciones_df(df_ins, "cuit", "nombre")
print(f"\n🔸 cuit → nombre")
print(f"   Violaciones: {n2} CUIT con >1 nombre")
if n2 > 0:
    cuit_ejemplo = det2.index[0]
    print(f"\n   Ejemplo: CUIT '{cuit_ejemplo}' aparece con los nombres:")
    nombres_conflicto = df_ins.loc[df_ins["cuit"] == cuit_ejemplo,
                                    ["id_credito", "nombre", "_patologia"]]
    print(nombres_conflicto.to_string(index=False))

### 📝 Interpretación

La detección de violaciones de dependencia funcional es un *group-by + count distinct*, operación de costo $O(n)$ tras indexación. La lectura de los resultados exige criterio:

- **CP → provincia**: una violación puede deberse (a) a un CP mal cargado, o (b) a una reasignación administrativa real (cambios de jurisdicción son raros pero ocurren). La fuente autoritativa en Argentina es la tabla oficial de códigos postales de Correo Argentino; el pipeline debe incluir un *lookup* contra esa tabla.
- **CUIT → nombre**: dos nombres distintos bajo el mismo CUIT pueden indicar (a) error de digitación del CUIT, (b) error en el nombre, o (c) usurpación de identidad. En finanzas, esta tercera posibilidad activa alertas de cumplimiento PLAFT/UIF y exige investigación manual inmediata.

## 10. Etapa 3 — Restricciones de integridad cruzada

Las restricciones $\phi(a_{i1}, \ldots, a_{ip}) = \text{TRUE}$ vinculan lógicamente varios atributos del mismo registro. Implementamos cuatro del apunte como *assertions* programáticas.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 10. INTEGRIDAD CRUZADA
# ══════════════════════════════════════════════════════════════

# Las fechas ya están como datetime; verificamos el orden temporal.
df_ins["fecha_otorgamiento"] = pd.to_datetime(df_ins["fecha_otorgamiento"])
df_ins["fecha_vencimiento"]  = pd.to_datetime(df_ins["fecha_vencimiento"])

reglas_cruzadas = {
    "venc_posterior_a_otorg":
        lambda d: d["fecha_vencimiento"] > d["fecha_otorgamiento"],

    "monto_positivo_si_otorgado":
        lambda d: (d["credit_amount"] > 0) | d["fecha_otorgamiento"].isna(),

    "ingreso_coherente_con_monto":
        lambda d: d["ingreso_mensual"] * d["duration"] >= 0.1 * d["credit_amount"],
        # cuota máx. 10% de ingreso × duración debe cubrir algo del monto

    "empleado_con_ingreso_positivo":
        lambda d: (d["tipo_empleo"].isin(["jubilado"]) |
                   (d["ingreso_mensual"] > 0)),
}

log_cruz = []
for nombre_regla, fn in reglas_cruzadas.items():
    mask_ok = fn(df_ins)
    n_viol = (~mask_ok).sum()
    log_cruz.append({"regla": nombre_regla,
                     "violaciones": n_viol,
                     "pct": f"{100*n_viol/len(df_ins):.2f}%"})

df_log_cruz = pd.DataFrame(log_cruz)
print("🔍 Log de integridad cruzada:\n")
print(df_log_cruz.to_string(index=False))

In [ ]:
# --- Inspección del caso patológico: venc < otorg -------------------
mask_temp = df_ins["fecha_vencimiento"] <= df_ins["fecha_otorgamiento"]
print(f"Registros con vencimiento ≤ otorgamiento: {mask_temp.sum()}")
if mask_temp.sum() > 0:
    print("\n📋 Muestra:")
    cols = ["id_credito", "fecha_otorgamiento", "fecha_vencimiento",
            "duration", "_patologia"]
    print(df_ins.loc[mask_temp, cols].head().to_string(index=False))

### 📝 Interpretación

Las reglas cruzadas **codifican el conocimiento del negocio en lenguaje formal**. Dos beneficios clave:

- **Reproducibilidad**: la regla queda en código, versionada en Git, reproducible en cualquier ambiente. Un *experto* que "mira los datos" no escala; un script que evalúa 50 reglas sobre 10 millones de registros, sí.
- **Documentación ejecutable**: cada regla es a la vez especificación y verificación. Cuando un área de negocio propone cambiar una regla, el impacto se mide ejecutando el script antes y después.

La distinción *hard* vs. *soft* del apunte es práctica: `venc ≤ otorg` es **hard** (imposible); `ingreso × duration < 10% × monto` es **soft** (implausible pero no imposible, puede ocurrir en refinanciaciones).

## 11. Etapa 4a — Detección estadística: z-score modificado con MAD

El z-score clásico $z_i = (x_i - \bar{x})/s$ se ve afectado por los propios outliers que pretende detectar (la media y el desvío están **contaminados**). El z-score modificado emplea estimadores robustos:

$$
M_i = \frac{0{,}6745 \cdot (x_i - \widetilde{x})}{\text{MAD}}, \qquad \text{MAD} = \text{mediana}(|x_i - \widetilde{x}|)
$$

con punto de quiebre del 50% (vs. 0% del z-score clásico). Regla de decisión: $|M_i| > 3{,}5$ es sospechoso (Iglewicz & Hoaglin, 1993).

In [ ]:
# ══════════════════════════════════════════════════════════════
# 11. Z-SCORE MODIFICADO BASADO EN MAD
# ══════════════════════════════════════════════════════════════

def z_score_modificado(x):
    """Calcula el z-score modificado de Iglewicz–Hoaglin (1993).
    Robusto a outliers: emplea mediana y MAD en lugar de media y DE.
    """
    x = np.asarray(x, dtype=float)
    mediana = np.nanmedian(x)
    mad = np.nanmedian(np.abs(x - mediana))
    if mad == 0:
        return np.zeros_like(x)
    return 0.6745 * (x - mediana) / mad

# Aplicación a ingreso_mensual (variable contaminada con los valores absurdos)
df_ins["z_ingreso"] = z_score_modificado(df_ins["ingreso_mensual"])

# Detección
UMBRAL_MAD = 3.5
df_ins["outlier_mad"] = np.abs(df_ins["z_ingreso"]) > UMBRAL_MAD

# Comparación con z-score clásico (para resaltar la diferencia)
x = df_ins["ingreso_mensual"].values
df_ins["z_clasico"] = (x - np.nanmean(x)) / np.nanstd(x)
df_ins["outlier_clasico"] = np.abs(df_ins["z_clasico"]) > 3

tabla_comp = pd.DataFrame({
    "Método": ["Z clásico", "Z modificado (MAD)"],
    "Outliers detectados": [df_ins["outlier_clasico"].sum(),
                            df_ins["outlier_mad"].sum()],
})
print("📊 Comparación entre z-score clásico y z-score modificado:\n")
print(tabla_comp.to_string(index=False))

In [ ]:
# --- Visualización: ventaja del z-score modificado ------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Izquierda: histograma de ingreso, escala log + líneas de umbral
x_safe = df_ins["ingreso_mensual"].clip(lower=1)
axes[0].hist(np.log10(x_safe), bins=60, color=COL_TEAL, alpha=0.8,
             edgecolor="white")
axes[0].set_title("Distribución de log10(ingreso_mensual)",
                  color=COL_NAVY, fontweight="bold")
axes[0].set_xlabel("log10(ingreso)")
axes[0].set_ylabel("Frecuencia")

# Derecha: scatter z-clásico vs z-modificado
axes[1].scatter(df_ins["z_clasico"], df_ins["z_ingreso"],
                alpha=0.4, s=14, color=COL_PURPLE)
axes[1].axhline( 3.5, color=COL_RED, linestyle="--", lw=1, label="|M| = 3.5")
axes[1].axhline(-3.5, color=COL_RED, linestyle="--", lw=1)
axes[1].axvline( 3,   color=COL_AMBER, linestyle="--", lw=1, label="|z| = 3")
axes[1].axvline(-3,   color=COL_AMBER, linestyle="--", lw=1)
axes[1].set_xlabel("z clásico"); axes[1].set_ylabel("z modificado (MAD)")
axes[1].set_title("Comparación entre estimadores",
                  color=COL_NAVY, fontweight="bold")
axes[1].legend()

plt.tight_layout(); plt.show()

### 📝 Interpretación

El gráfico derecho revela el fenómeno central: los outliers que insertamos son tan extremos (ingresos de $10^{10}$ pesos) que inflan la desviación estándar clásica hasta volverla casi inútil. Con la DE inflada, muchos valores genuinamente extremos pero legítimos quedan *por debajo* del umbral $|z|>3$ y se dejan pasar; el z-score modificado, indiferente a esos extremos por construcción, los detecta sin dificultad.

Esta propiedad —punto de quiebre del 50%— es especialmente valiosa en banca privada, donde los clientes *high net worth* constituyen una cola legítima de la distribución de ingresos que el método clásico confundiría con errores.

## 12. Etapa 4b — Detección estadística: IQR contextualizado

Un ingreso de \$2.000.000 mensuales es atípico para un empleado público de Misiones pero plausible para un ejecutivo del sector financiero de CABA. La detección debe **contextualizarse por segmento**: calculamos IQR y umbrales dentro de cada combinación `provincia × tipo_empleo`.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 12. IQR CONTEXTUALIZADO POR SEGMENTO
# ══════════════════════════════════════════════════════════════

def detectar_iqr_segmento(df, var, by, k=1.5):
    """Detecta outliers IQR dentro de cada segmento `by`.
    Retorna una Series booleana alineada con el índice de df.
    """
    out = pd.Series(False, index=df.index)
    for clave, grupo in df.groupby(by):
        q1, q3 = grupo[var].quantile([0.25, 0.75])
        iqr = q3 - q1
        lo, hi = q1 - k * iqr, q3 + k * iqr
        mask_seg = (grupo[var] < lo) | (grupo[var] > hi)
        out.loc[grupo.index] = mask_seg
    return out

# Filtrado previo: trabajamos sólo con registros cuyo dominio es válido
# (descartamos negativos y absurdos que ya fueron identificados arriba).
df_iqr = df_ins.query("0 <= ingreso_mensual < 1e9").copy()

df_iqr["outlier_iqr_global"] = detectar_iqr_segmento(
    df_iqr, "ingreso_mensual", by=lambda i: "global"
)
df_iqr["outlier_iqr_ctx"] = detectar_iqr_segmento(
    df_iqr, "ingreso_mensual", by=["provincia", "tipo_empleo"]
)

print("🔍 Comparación IQR global vs contextualizado:")
print(f"   Outliers IQR global         : {df_iqr['outlier_iqr_global'].sum()}")
print(f"   Outliers IQR contextualizado: {df_iqr['outlier_iqr_ctx'].sum()}")

# Desagregación por segmento
print("\n📊 Outliers por segmento (IQR contextualizado):")
tabla_seg = (df_iqr.groupby(["provincia", "tipo_empleo"])
             .agg(n=("ingreso_mensual", "size"),
                  outliers=("outlier_iqr_ctx", "sum"))
             .query("outliers > 0")
             .sort_values("outliers", ascending=False))
print(tabla_seg.head(10).to_string())

In [ ]:
# --- Visualización: IQR por segmento --------------------------------
fig, ax = plt.subplots(figsize=(14, 6))
segmentos = (df_iqr.groupby(["provincia", "tipo_empleo"])["ingreso_mensual"]
             .median().sort_values().reset_index())
top_seg = segmentos.tail(12)

data = [df_iqr.query("provincia==@p and tipo_empleo==@t")["ingreso_mensual"]
        for p, t in zip(top_seg["provincia"], top_seg["tipo_empleo"])]
labels = [f"{p[:4]}/{t[:6]}" for p, t in zip(top_seg["provincia"],
                                              top_seg["tipo_empleo"])]

bp = ax.boxplot(data, labels=labels, patch_artist=True, showfliers=True)
for patch in bp["boxes"]:
    patch.set_facecolor(COL_TEAL); patch.set_alpha(0.7)
ax.set_yscale("log")
ax.set_ylabel("Ingreso mensual (ARS, escala log)")
ax.set_title("Distribución del ingreso por segmento (provincia × empleo)",
             color=COL_NAVY, fontweight="bold")
plt.xticks(rotation=45, ha="right")
plt.tight_layout(); plt.show()

### 📝 Interpretación

El IQR contextualizado produce detecciones **más precisas y menos heterogéneas**: un ingreso que a nivel agregado parecería atípico queda re-clasificado como típico dentro de su segmento, y viceversa. El costo es mayor número de comparaciones (una distribución por celda de la grilla `provincia × tipo_empleo`), y la necesidad de que cada celda tenga suficiente masa para que Q1, Q3 e IQR sean estables. En la práctica se combinan: IQR global como primer filtro grueso, IQR contextualizado para segmentos con $n$ suficiente.

---

## 13. Etapa 5 — Tratamiento y protocolo integral de 5 etapas

Sintetizamos todo el trabajo previo en un **pipeline secuencial reproducible** que replica el protocolo del apunte:

| Etapa | Operación | Resultado |
|---|---|---|
| 1 | Validación de dominio | Log de violaciones de $\mathcal{D}_j$ |
| 2 | Dependencias funcionales | Log de violaciones de $A \to B$ |
| 3 | Integridad cruzada | Log de violaciones de $\phi$ |
| 4 | Detección estadística | Log de outliers (MAD + IQR) |
| 5 | Tratamiento | Dataset corregido + log de auditoría |

Cada etapa alimenta un **log de auditoría único** con las cuatro dimensiones del principio de trazabilidad: valor original, regla violada, acción tomada, justificación.

In [ ]:
# ══════════════════════════════════════════════════════════════
# 13. PIPELINE INTEGRAL + LOG DE AUDITORÍA
# ══════════════════════════════════════════════════════════════

class AuditLog:
    """Registrador del pipeline de limpieza. Conforma el log de
    auditoría requerido por el principio de trazabilidad (BCRA,
    Basilea III, Modelos Internos)."""
    def __init__(self):
        self.entradas = []

    def agregar(self, id_reg, variable, valor_orig,
                regla, accion, valor_nuevo, justificacion):
        self.entradas.append({
            "timestamp": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
            "id_registro": id_reg,
            "variable": variable,
            "valor_original": valor_orig,
            "regla_violada": regla,
            "accion": accion,
            "valor_nuevo": valor_nuevo,
            "justificacion": justificacion,
        })

    def to_frame(self):
        return pd.DataFrame(self.entradas)


audit = AuditLog()
df_clean = df_dedup_exacto.copy()

# ----- Etapa 1: dominio (ingresos anómalos) -------------------------
for idx, row in df_clean.iterrows():
    if row["ingreso_mensual"] < 0:
        orig = row["ingreso_mensual"]
        df_clean.at[idx, "ingreso_mensual"] = abs(orig)
        audit.agregar(row["id_credito"], "ingreso_mensual", orig,
                      "dominio: ingreso >= 0",
                      "corregir signo", abs(orig),
                      "Error de signo: se asume digitación incorrecta")
    elif row["ingreso_mensual"] > 1e9:
        orig = row["ingreso_mensual"]
        df_clean.at[idx, "ingreso_mensual"] = np.nan
        audit.agregar(row["id_credito"], "ingreso_mensual", orig,
                      "dominio: ingreso < 1e9",
                      "marcar NA", np.nan,
                      "Valor económicamente implausible; "
                      "derivado a imputación")

# ----- Etapa 3: integridad cruzada (fechas invertidas) --------------
for idx, row in df_clean.iterrows():
    if pd.to_datetime(row["fecha_vencimiento"]) <= pd.to_datetime(
            row["fecha_otorgamiento"]):
        orig = row["fecha_vencimiento"]
        nueva = (pd.to_datetime(row["fecha_otorgamiento"])
                 + pd.DateOffset(months=int(row["duration"])))
        df_clean.at[idx, "fecha_vencimiento"] = nueva
        audit.agregar(row["id_credito"], "fecha_vencimiento", orig,
                      "integridad: venc > otorg",
                      "recalcular desde otorg + duration", nueva,
                      "Violación temporal: fecha recomputada "
                      "deterministamente")

# ----- Etapa 2: dependencia funcional CP → provincia -----------------
# Para cada provincia válida, si un CP no cae en su rango, se marca.
for idx, row in df_clean.iterrows():
    prov, cp = row["provincia"], row["cod_postal"]
    try:
        cp_int = int(cp)
        lo, hi = PROVINCIAS_CP[prov]
        if not (lo <= cp_int <= hi):
            orig = cp
            df_clean.at[idx, "cod_postal"] = np.nan
            audit.agregar(row["id_credito"], "cod_postal", orig,
                          "dep_func: CP → provincia",
                          "marcar NA", np.nan,
                          f"CP '{orig}' no corresponde a provincia '{prov}'")
    except Exception:
        pass

df_audit = audit.to_frame()
print(f"✅ Pipeline ejecutado. Registros en log de auditoría: "
      f"{len(df_audit):,}")
print("\n📋 Distribución de acciones:")
print(df_audit["accion"].value_counts().to_string())

In [ ]:
# --- Muestra del log de auditoría -----------------------------------
print("📋 Primeras 10 entradas del log de auditoría:\n")
with pd.option_context("display.max_colwidth", 40):
    print(df_audit.head(10).to_string(index=False))

In [ ]:
# --- Resumen comparativo: pre vs. post pipeline ---------------------
resumen = pd.DataFrame({
    "Dataset sucio":  [len(df_sucio),
                       (df_sucio["ingreso_mensual"] < 0).sum(),
                       (df_sucio["ingreso_mensual"] > 1e9).sum(),
                       df_sucio["ingreso_mensual"].median(),
                       df_sucio["credit_amount"].mean()],
    "Tras dedup":     [len(df_dedup_exacto),
                       (df_dedup_exacto["ingreso_mensual"] < 0).sum(),
                       (df_dedup_exacto["ingreso_mensual"] > 1e9).sum(),
                       df_dedup_exacto["ingreso_mensual"].median(),
                       df_dedup_exacto["credit_amount"].mean()],
    "Tras pipeline":  [len(df_clean),
                       (df_clean["ingreso_mensual"] < 0).sum(),
                       (df_clean["ingreso_mensual"] > 1e9).sum(),
                       df_clean["ingreso_mensual"].median(),
                       df_clean["credit_amount"].mean()],
}, index=["N registros",
          "Ingresos negativos",
          "Ingresos > 10⁹",
          "Mediana ingreso",
          "Media credit_amount"])

print("📊 Trayectoria del dataset a lo largo del pipeline:\n")
print(resumen.round(2).to_string())

### 📝 Interpretación

El pipeline deja **trazabilidad completa**: cada decisión automatizada queda registrada con sus cuatro atributos obligatorios. Esta trazabilidad no es un lujo académico sino un requisito regulatorio:

- **Normativa BCRA** (Com. "A" 6060, modelos internos de capital): las instituciones deben demostrar que los datos de entrada de sus modelos han sido procesados con reglas documentadas y reproducibles.
- **Basilea III / IV**: los modelos avanzados (IRB) exigen auditoría trimestral de las transformaciones aplicadas a los datos históricos.
- **ISO 8000 (data quality)**: la trazabilidad de cambios es uno de los pilares de la certificación.

En producción, este log se persiste en un *data warehouse* separado, con retención mínima de 7 años y acceso auditable.

---

## 14. Síntesis y conexión con el apunte teórico

Hemos instanciado en código la tabla comparativa del apunte:

| Dimensión | Duplicados | Inconsistencias |
|---|---|---|
| **Naturaleza** | Redundancia | Contradicción |
| **Detección** | Levenshtein, Jaro-Winkler, Fellegi-Sunter, blocking | Validación de dominio, dep. funcional, z-MAD, IQR |
| **Complejidad** | $O(n^2)$ → $O(n)$ con blocking | $O(n \cdot p)$ determinista; $O(n \cdot p \cdot S)$ contextualizado |
| **Tratamiento** | Deduplicación, golden record | Corrección, imputación, NA, eliminación |
| **Impacto** | Sesgo en frecuencias, inflación de cartera | Coeficientes distorsionados, predicciones aberrantes |

### Tres aprendizajes centrales

1. **La duplicación y la inconsistencia son fenómenos ortogonales.** Un registro duplicado puede ser íntegro (todas sus copias son coherentes); un registro inconsistente puede ser único. Exigen tratamientos distintos y ambos deben aplicarse secuencialmente.

2. **La calidad del dato antecede a la modelización.** Ningún algoritmo —por sofisticado que sea— compensa la basura de entrada (*garbage in, garbage out*). El tiempo invertido en limpieza rinde más que el tiempo invertido en afinar hiperparámetros.

3. **La trazabilidad es innegociable en finanzas.** Toda transformación debe ser auditable. Este notebook ha escrito cada decisión al log, sentando la base para cumplimiento normativo.

### Próximos pasos en el curso

- **Clase 3**: outliers y valores faltantes, profundizando la detección estadística avanzada con métodos multivariados (Mahalanobis, Isolation Forest).
- **Clase 4**: feature engineering, VIF, selección de variables para scoring crediticio.
- **Clase 5 en adelante**: modelos de clasificación sobre el dataset limpio producto de estas primeras clases.

---

## 15. Referencias bibliográficas

1. Fellegi, I. P. & Sunter, A. B. (1969). *A Theory for Record Linkage*. **JASA**, 64(328), 1183–1210.
2. Winkler, W. E. (1990). *String Comparator Metrics and Enhanced Decision Rules in the Fellegi-Sunter Model of Record Linkage*. Proceedings of the Section on Survey Research Methods, AMS.
3. Christen, P. (2012). *Data Matching: Concepts and Techniques for Record Linkage, Entity Resolution, and Duplicate Detection*. Springer.
4. Iglewicz, B. & Hoaglin, D. C. (1993). *How to Detect and Handle Outliers*. ASQC Quality Press.
5. Levenshtein, V. I. (1966). *Binary Codes Capable of Correcting Deletions, Insertions, and Reversals*. Soviet Physics Doklady, 10(8), 707–710.
6. Rahm, E. & Do, H. H. (2000). *Data Cleaning: Problems and Current Approaches*. IEEE Data Engineering Bulletin, 23(4), 3–13.
7. Fan, W. & Geerts, F. (2012). *Foundations of Data Quality Management*. Morgan & Claypool.
8. Díaz, D. E. (2026). *Apunte Teórico Clase 2 — Registros Duplicados e Inconsistencias en Datos Financieros*. MMD, UTN FRRo.

---

<div align="center">

**Dr. Darío Ezequiel Díaz** · Aplicaciones de Minería de Datos a Economía y Finanzas
Maestría en Minería de Datos · UTN Facultad Regional Rosario · 2026

</div>